<a href="https://colab.research.google.com/github/enzomiguel7/missao_espacial_telemetria/blob/main/Global_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Conteúdo do dados.cvs**
## Categoria,Subcategoria,Valor,Unidade
- Categoria;Subcategoria;Valor;Unidade
- Status;Motores;1;
- Status;Sistema de Voo;1;
- Status;Suporte à Vida;1;
- Status;Escudo Térmico;1;
- Status;Geradores de Energia Solar;1;
- Status;Sistemas de Comunicação;1;
- Status;Laboratório Científico;0;
- Status;Módulo de Agricultura;0;
- Status;Módulo Médico;0;

---

- Energia;0800;-500;kWh
- Energia;1200;-200;kWh
- Energia;2000;-150;kWh
- Energia;1200;600;kWh
- Energia;2000;80;kWh
- Energia;2300;500;kWh

---

- Ambiente;Temperatura Externa na Terra;25;°C
- Ambiente;Temperatura Externa no Espaço;-3;°C
- Ambiente;Temperatura Interna dos Tanques;-190;°C
- Ambiente;Temperatura Interna da Nave;22;°C
- Ambiente;Nível de Oxigênio;21;%
- Ambiente;Nível de CO2;500;ppm
- Ambiente;Qualidade de Comunicação;72;%
- Ambiente;Vento na Terra;4;m/s

---

- Log;Alerta;Oscilação na comunicação às 02:15;
- Log;Reinicialização;Sistema de voo reiniciado às 02:30;
- Log;Falha;Sensor de radiação não respondeu às 03:05;
- Log;Economia;Modo de baixo consumo ativado às 03:20;
- Log;Prioridade;Suporte à vida definido como prioridade máxima às 03:45;
- Log;Alerta;Pico inesperado de consumo energético às 04:00;
- Log;Reinicialização;Laboratório científico inicializado às 04:15;
- Log;Inconsistência;Reserva energética registrada como -120 kWh às 04:30;
- Log;Falha;Escudo térmico não respondeu ao teste às 05:00;
- Log;Alerta;Nível de oxigênio caiu para 19% às 05:15;

---

- Reserva;0800;92;%
- Reserva;1100;80;%
- Reserva;1400;69;%
- Reserva;1700;56;%
- Reserva;2000;44;%
- Reserva;2300;32;%

In [10]:
conteudo_csv = """Status,Motores,1
Status,Sistema de Voo,1
Status,Suporte à Vida,1
Status,Escudo Térmico,1
Status,Geradores de Energia Solar,1
Status,Sistemas de Comunicação,1
Status,Laboratório Científico,0
Status,Módulo de Agricultura,0
Status,Módulo Médico,0
Energia,800,-500,kWh
Energia,1200,-200,kWh
Energia,2000,-150,kWh
Energia,1200,600,kWh
Energia,2000,80,kWh
Energia,2300,500,kWh
Ambiente,Temperatura Externa na Terra,25,°C
Ambiente,Temperatura Externa no Espaço,-3,°C
Ambiente,Temperatura Interna dos Tanques,-190,°C
Ambiente,Temperatura Interna da Nave,22,°C
Ambiente,Nível de Oxigênio,21,%
Ambiente,Nível de CO2,500,ppm
Ambiente,Qualidade de Comunicação,72,%
Ambiente,Vento na Terra,4,m/s
Log,Alerta,Oscilação na comunicação às 02:15
Log,Reinicialização,Sistema de voo reiniciado às 02:30
Log,Falha,Sensor de radiação não respondeu às 03:05
Log,Economia,Modo de baixo consumo ativado às 03:20
Log,Prioridade,Suporte à vida definido como prioridade máxima às 03:45
Log,Alerta,Pico inesperado de consumo energético às 04:00
Log,Reinicialização,Laboratório científico inicializado às 04:15
Log,Inconsistência,Reserva energética registrada como -120 kWh às 04:30
Log,Falha,Escudo térmico não respondeu ao teste às 05:00
Log,Alerta,Nível de oxigênio caiu para 19% às 05:15
Reserva,800,92
Reserva,1100,80
Reserva,1400,69
Reserva,1700,56
Reserva,2000,44
Reserva,2300,32"""

with open("dados.csv", "w", encoding="utf-8") as f:
    f.write(conteudo_csv)

print("Arquivo criado com sucesso!")

Arquivo criado com sucesso!


In [11]:
class Painel:
    def __init__(self, conjunto_de_dados):
        self.status_modulos = {}
        self.historico_energia = []
        self.variaveis_ambientais = {}
        self.fila_alertas = []
        self.pilha_criticos = []
        self.historico_reserva = []

        self.carregar_dados(conjunto_de_dados)

    def carregar_dados(self, caminho_arquivo):
        with open(caminho_arquivo, "r", encoding="utf-8") as arquivo:
            for linha in arquivo:
                linha = linha.strip()

                if not linha or linha.startswith('#') or linha.startswith('Categoria'):
                    continue

                parts = linha.split(',')
                categoria = parts[0]

                if categoria == "Status":
                    nome_modulo = parts[1]
                    valor = int(parts[2])
                    self.status_modulos[nome_modulo] = valor

                elif categoria == "Energia":
                    horario = parts[1]
                    valor = float(parts[2])
                    unidade = parts[3]
                    self.historico_energia.append((horario, valor, unidade))

                elif categoria == "Ambiente":
                    nome_variavel = parts[1]
                    valor = float(parts[2])
                    self.variaveis_ambientais[nome_variavel] = valor

                elif categoria == "Log":
                    tipo_log = parts[1]
                    mensagem = parts[2]
                    texto_completo = f"{tipo_log}: {mensagem}"

                    if tipo_log in ["Alerta", "Reinicialização", "Economia", "Prioridade"]:
                        self.fila_alertas.append(texto_completo)
                    elif tipo_log in ["Falha", "Inconsistência"]:
                        self.pilha_criticos.append(texto_completo)

                elif categoria == "Reserva":
                    horario = parts[1]
                    valor = float(parts[2])
                    self.historico_reserva.append((horario, valor))

    def calcular_previsao(self):
        reserva = [item[1] for item in self.historico_reserva]

        if len(reserva) < 2:
            return 0, 0

        medicoes = list(range(1, len(reserva) + 1))

        def regressao_linear(xs, ys):
            n = len(xs)
            media_x = sum(xs) / n
            media_y = sum(ys) / n

            numerador = 0
            denominador = 0

            for x, y in zip(xs, ys):
                numerador += (x - media_x) * (y - media_y)
                denominador += (x - media_x) ** 2

            if denominador == 0:
                return 0, media_y

            coeficiente_a = numerador / denominador
            coeficiente_b = media_y - coeficiente_a * media_x
            return coeficiente_a, coeficiente_b

        def prever(coef_a, coef_b, x_futuro):
            return coef_a * x_futuro + coef_b

        a, b = regressao_linear(medicoes, reserva)
        proxima_medicao = len(reserva) + 1
        reserva_prevista = prever(a, b, proxima_medicao)

        return a, reserva_prevista

    def processar_regras(self, reserva_prevista):
        oxigenio = self.variaveis_ambientais.get("Nível de Oxigênio", 21)
        co2 = self.variaveis_ambientais.get("Nível de CO2", 500)
        temp_nave = self.variaveis_ambientais.get("Temperatura Interna da Nave", 22)

        suporte_vida = self.status_modulos.get("Suporte à Vida", 1)
        motores = self.status_modulos.get("Motores", 1)
        escudo = self.status_modulos.get("Escudo Térmico", 1)

        crise_suporte = (not suporte_vida) or (oxigenio < 19.5 and co2 > 600)

        crise_mecanica = (motores == 0) or (escudo == 0) or (temp_nave > 35 or temp_nave < 15)

        crise_bateria_futura = (reserva_prevista < 40.0)


        if len(self.pilha_criticos) > 0 or crise_suporte or crise_mecanica:
            status_geral = "CRÍTICO"
        elif crisis_bateria_futura or oxigenio < 20.0 or co2 > 550:
            status_geral = "ALERTA"
        else:
            status_geral = "NORMAL"

        return status_geral

    def gerar_painel(self, status_geral, taxa_queda, reserva_prevista):
        print("============================================================")
        print("🛰️ PAINEL OPERACIONAL - MISSÃO ESPACIAL EXPERIMENTAL")
        print("============================================================")
        print(f"STATUS GERAL DA MISSÃO: {status_geral}")
        print("------------------------------------------------------------")

        print("SEÇÃO 1: EMERGÊNCIAS CRÍTICAS (ORGANIZADO POR PILHA)")
        if self.pilha_criticos:
            ultimo_erro = self.pilha_criticos.pop()
            print(f"-> AÇÃO IMEDIATA REQUERIDA: {ultimo_erro}")
            if self.pilha_criticos:
                print(f"-> Outros erros aguardando na pilha: {len(self.pilha_criticos)}")
        else:
            print("-> Nenhuma emergência crítica registrada na pilha.")
        print("------------------------------------------------------------")

        print("SEÇÃO 2: HISTÓRICO DE AVISOS OPERACIONAIS (ORGANIZADO POR FILA)")
        if self.fila_alertas:
            for i, alerta in enumerate(self.fila_alertas, 1):
                print(f"{i}º na Fila: {alerta}")
        else:
            print("-> Fila de alertas vazia.")
        print("------------------------------------------------------------")

        print("SEÇÃO 3: ANÁLISE E PREVISÃO DE DADOS (REGRESSÃO LINEAR)")
        print(f"-> Ritmo de queda: A reserva de energia cai {round(taxa_queda * -1, 2)}% por medição.")
        print(f"-> Projeção para o próximo ciclo: A reserva cairá para {round(reserva_prevista, 2)}%.")
        print("------------------------------------------------------------")

        print("SEÇÃO 4: RECOMENDAÇÕES TÉCNICAS DO SISTEMA")
        if status_geral == "CRÍTICO":
            print("1. [CRÍTICA] Interromper pesquisas e isolar energia para o Suporte à Vida.")
            print("2. [CRÍTICA] Executar diagnóstico manual no módulo apontado pelo topo da pilha.")
            print("3. [ALTA] Redirecionar carga dos Geradores Solares exclusivamente para as baterias.")
        elif status_geral == "ALERTA":
            print("1. [ALTA] Ativar modo econômico nos módulos secundários (Laboratório e Agricultura).")
            print("2. [MÉDIA] Monitorar estabilização dos sensores ambientais nos próximos ciclos.")
        else:
            print("1. [BAIXA] Manter plano de voo padrão.")
            print("2. [BAIXA] Prosseguir com atividades laboratoriais agendadas.")
        print("============================================================")


# Bloco Principal de Execução
if __name__ == "__main__":
    # 1. Define o caminho do arquivo de dados
    caminho_do_csv = "dados.csv"

    # 2. Instancia a classe (o construtor roda e já carrega os dados)
    painel_missao = Painel(caminho_do_csv)

    # 3. Executa o cálculo matemático da previsão da reserva
    taxa_queda, reserva_projetada = painel_missao.calcular_previsao()

    # 4. Processa as regras de decisão passando o valor que foi previsto
    status_final = painel_missao.processar_regras(reserva_projetada)

    # 5. Renderiza e imprime toda a interface do painel na tela
    painel_missao.gerar_painel(status_final, taxa_queda, reserva_projetada)

🛰️ PAINEL OPERACIONAL - MISSÃO ESPACIAL EXPERIMENTAL
STATUS GERAL DA MISSÃO: CRÍTICO
------------------------------------------------------------
SEÇÃO 1: EMERGÊNCIAS CRÍTICAS (ORGANIZADO POR PILHA)
-> AÇÃO IMEDIATA REQUERIDA: Falha: Escudo térmico não respondeu ao teste às 05:00
-> Outros erros aguardando na pilha: 2
------------------------------------------------------------
SEÇÃO 2: HISTÓRICO DE AVISOS OPERACIONAIS (ORGANIZADO POR FILA)
1º na Fila: Alerta: Oscilação na comunicação às 02:15
2º na Fila: Reinicialização: Sistema de voo reiniciado às 02:30
3º na Fila: Economia: Modo de baixo consumo ativado às 03:20
4º na Fila: Prioridade: Suporte à vida definido como prioridade máxima às 03:45
5º na Fila: Alerta: Pico inesperado de consumo energético às 04:00
6º na Fila: Reinicialização: Laboratório científico inicializado às 04:15
7º na Fila: Alerta: Nível de oxigênio caiu para 19% às 05:15
------------------------------------------------------------
SEÇÃO 3: ANÁLISE E PREVISÃO DE DA

# **Conteúdo do README.md**
# Equipe
- Enzo Miguel de Oliveira Silva - rm571639
- Gabriel da Silva Amaral - rm570421
- Giovanni Dutra Dias - rm572671
- Pedro Aulicino Corrêa Coelho - rm570804
- Pedro Henrique Benício Santana Braga - rm572066

# Introdução ao Problema
Criar um simulador de um painel de controle para uma missão espacial.

# Dados Utilizados

O arquivo "dados.csv" contém informações organizadas em cinco categorias principais:

1. **Status dos Módulos:** Indica se cada sistema crítico está ativo ou inativo.

2. **Telemetria de Energia:** Registra consumo e geração de energia durante diferentes fases da missão.

3. **Variáveis Ambientais:** Reúne dados como temperatura, oxigênio, CO₂, comunicação e ventos.

4. **Log de Eventos:** Lista alertas e falhas para análise de erros e decisões.

5. **Reserva de Energia:** Porcentagem de energia disponível, usada para prever a tendência de queda.

# Regras Lógicas do Sistema
Para avaliar a segurança e definir o estado operacional da missão, o sistema aplica testes lógicos baseados nos dados coletados:

O programa cruza dados de atmosfera e temperatura usando operadores lógicos como AND, OR e NOT.

Uma falha é detectada se o suporte à vida desligar, se os gases saírem da faixa segura ou se algum motor falhar.

O sistema analisa a Pilha de Críticos e a previsão de bateria para definir o status final como NORMAL, ALERTA ou CRÍTICO.


# Técnica de Previsão
Para prever a tendência de queda da reserva de energia, foi usado um algoritmo simples de regressão linear:
- Os valores históricos de reserva são extraídos do arquivo dados.csv.
- A regressão calcula a taxa média de variação da energia a cada medição.
- Com base nessa tendência, o algoritmo estima o percentual de energia disponível em medições futuras.

# Recomendações do Sistema
Com base no status geral definido pelas regras, o painel gera ações automáticas.

Status Crítico: Ordena o corte imediato de energia de setores secundários (como laboratórios) e exige manutenção urgente no erro do topo da pilha.

Status de Alerta: Prescreve ações preventivas, como ativar modos de economia de bateria e monitorar sensores mais de perto.

Status Normal: Autoriza a tripulação a manter o plano de voo padrão e a rotina normal de pesquisas científicas.

# Como Executar o Código
 - Clique no botão "play" na primeira célula para gerar os dados
 - Clique no botão "play" na segunda célula para gerar a interface da telemetria.

# Link para o Vídeo

# Conclusão
O desenvolvimento deste sistema inteligente de monitoramento demonstrou a importância do pensamento computacional e das estruturas de dados em cenários de missão crítica. Através da divisão correta de responsabilidades entre listas, dicionários, filas e pilhas, o software foi capaz de processar telemetrias espaciais complexas em tempo real. A integração do algoritmo de regressão linear provou como a análise preditiva pode antecipar falhas de energia e guiar recomendações técnicas automatizadas, garantindo de forma ética e segura a integridade dos equipamentos e a sobrevivência da tripulação.